### DistilBERT Baseline Model

In [19]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline 
import pandas as pd

In [20]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df.head(5)

,business_id,text,individual_stars,date,user_id,name,city,state,total_business_stars,review_count,categories,cuisine
0,YB26JvvGS2LgkxEKOObSAw,i've been eating at this restaurant for over 5...,5.0,2021-01-08 01:49:36,iAD32p6h32eKDVxsPHSRHA,Unagi & Sushi,Metairie,LA,4.0,62,"['Sushi Bars', 'Restaurants', 'Japanese']",Japanese
1,jfIwOEXcVRyhZjM4ISOh4g,how does a delivery person from here get lost ...,1.0,2021-01-02 00:19:00,rYvWv-Ny16b1lMcw1IP7JQ,Oreland Pizza,Oreland,PA,3.0,29,"['Sandwiches', 'Restaurants', 'Burgers', 'Pizza']",American
2,yE1raqkLX7OZsjmX3qKIKg,two words: whipped. feta. \nexplosion of amazi...,5.0,2021-01-27 23:28:03,j4qNLF-VNRF2DwBkUENW-w,Butcher & Bee,Nashville,TN,4.0,863,"['Middle Eastern', 'American (New)', 'Restaura...",American
3,3PlpoDgeAQAreL8FM2LelA,place was great as well as parking. \nfood was...,4.0,2021-01-30 03:15:20,h41RIr5Rtkq7EoJ0tU0wgQ,Kauboi Izakaya,Reno,NV,4.0,312,"['Sushi Bars', 'Pubs', 'Gastropubs', 'Japanese...",Japanese
4,Rv6P37KiiuowrXti2JHZNQ,got this to go and it is for sure authentic! t...,5.0,2021-03-11 17:17:46,8_yoGifxsLHLRPtyDgMxhw,Nuevo Vallarta Authentic Mexican Food,Pinellas Park,FL,4.0,106,"['Event Planning & Services', 'Chicken Wings',...",Mexican


In [21]:
# DistilBERT sentiment pipeline
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0  
)

Device set to use mps:0


In [24]:
reviews_df["cuisine"].value_counts()

American    53487
Mexican     12773
Italian     11705
Japanese     8028
Chinese      6013
Indian       2296
Thai         2087
Greek        1601
Korean       1082
French        928
Name: cuisine, dtype: int64

In [25]:
def get_sentiment(texts, batch_size=64):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        outputs = sentiment_model(batch, truncation=True, max_length=512)
        results.extend(outputs)
    return results

results = get_sentiment(reviews_df['text'])

reviews_df['distilbert_label'] = [r['label'] for r in results]
reviews_df['distilbert_score'] = [r['score'] for r in results]

In [26]:
reviews_df.groupby('cuisine')['distilbert_score'].mean().sort_values()

cuisine
Chinese     0.984319
American    0.984524
Thai        0.985767
Mexican     0.985937
Italian     0.986303
Japanese    0.986756
Korean      0.987532
French      0.987844
Greek       0.988105
Indian      0.988792
Name: distilbert_score, dtype: float64

In [27]:
# Convert: POSITIVE score stays positive, NEGATIVE score flips to negative
reviews_df['distilbert_sentiment'] = reviews_df.apply(
    lambda x: x['distilbert_score'] if x['distilbert_label'] == 'POSITIVE' 
              else 1 - x['distilbert_score'], axis=1
)

reviews_df.groupby('cuisine')['distilbert_sentiment'].mean().sort_values()

cuisine
American    0.675329
Chinese     0.677033
Mexican     0.712524
Italian     0.725309
Japanese    0.737173
Greek       0.751363
Indian      0.775080
French      0.782882
Thai        0.786025
Korean      0.787356
Name: distilbert_sentiment, dtype: float64